# Binance USD-M Futures Partial Depth Service

This notebook shows how to call the local `BinanceFuturesDepthService` module from Jupyter. It subscribes to Binance USD-M Futures partial book depth streams, keeps the latest in-memory top-N depth snapshot per symbol, and reads bid1/bid2/ask1/ask2 from that table.

Run the cleanup cell at the end when you are done because the depth service starts a background receiver thread.

## 1. Import Local Package

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_path():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/home/suncong/binance_klines_data_fetch"),
    ]
    for candidate in candidates:
        if (candidate / "binance_klines_data_fetch").is_dir():
            return candidate.resolve()
    raise RuntimeError("Could not find the binance_klines_data_fetch repo path")


repo_path = find_repo_path()
repo_path_str = str(repo_path)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)

from binance_klines_data_fetch import BinanceDepthConfig, BinanceFuturesDepthService

print("Imported package from:", repo_path)

Imported package from: /home/suncong/binance_klines_data_fetch


## 2. Configure Partial Depth Streams

`levels` must be `5`, `10`, or `20`. `speed_ms` must be `100`, `250`, or `500`; `250` maps to the no-suffix stream name, for example `btcusdt@depth5`.

In [2]:
config = BinanceDepthConfig(
    symbols=["BTCUSDT", "ETHUSDT"],
    levels=5,
    speed_ms=100,
    read_timeout_seconds=30.0,
    startup_timeout_seconds=30.0,
)

service = BinanceFuturesDepthService(config)
print(service.build_url())

wss://fstream.binance.com/public/stream?streams=btcusdt@depth5@100ms/ethusdt@depth5@100ms


## 3. Start The Background Receiver

This opens one real Binance WebSocket connection. `block_until_ready=True` waits until the WebSocket connection is established and there is no current connection-level error. A specific symbol can still be missing a snapshot, so check `service.get_latest(symbol)` before using it.

In [3]:
service.start(block_until_ready=True, timeout=30.0)
status = service.status()
print("ready:", status.ready)
print("connected:", status.connected)
print("symbols:", status.symbols)

ready: True
connected: True
symbols: ('BTCUSDT', 'ETHUSDT')


## 4. Read Bid1/Bid2/Ask1/Ask2

In [4]:
def top_two_rows(snapshot):
    rows = []
    sample_time_ms = time.time_ns() // 1_000_000
    quote_age_ms = sample_time_ms - snapshot.event_time_ms

    for side_name, levels in [("bid", snapshot.bids[:2]), ("ask", snapshot.asks[:2])]:
        for level in levels:
            rows.append(
                {
                    "symbol": snapshot.symbol,
                    "side": side_name,
                    "level": level.level,
                    "price": level.price,
                    "qty": level.qty,
                    "event_time_ms": snapshot.event_time_ms,
                    "local_recv_time_ms": snapshot.local_recv_time_ms,
                    "quote_age_ms": quote_age_ms,
                    "receive_latency_ms": snapshot.receive_latency_ms,
                    "is_stale": snapshot.is_stale,
                    "sequence_gap": snapshot.sequence_gap,
                    "final_update_id": snapshot.final_update_id,
                }
            )
    return rows


rows = []
for symbol, snapshot in service.get_all_latest().items():
    if snapshot.is_stale or snapshot.sequence_gap:
        print(f"Skipping {symbol}: stale={snapshot.is_stale}, sequence_gap={snapshot.sequence_gap}")
        continue
    if len(snapshot.bids) < 2 or len(snapshot.asks) < 2:
        print(f"Skipping {symbol}: not enough depth levels")
        continue
    rows.extend(top_two_rows(snapshot))

display(pd.DataFrame(rows))

,symbol,side,level,price,qty,event_time_ms,local_recv_time_ms,quote_age_ms,receive_latency_ms,is_stale,sequence_gap,final_update_id
0,BTCUSDT,bid,1,73865.60,1.649,1780212719222,1780212719254,199,32,False,False,10670015611421
1,BTCUSDT,bid,2,73865.50,0.004,1780212719222,1780212719254,199,32,False,False,10670015611421
2,BTCUSDT,ask,1,73865.70,9.801,1780212719222,1780212719254,199,32,False,False,10670015611421
3,BTCUSDT,ask,2,73865.80,0.010,1780212719222,1780212719254,199,32,False,False,10670015611421
4,ETHUSDT,bid,1,2024.14,171.402,1780212719326,1780212719358,95,32,False,False,10670015615659
5,ETHUSDT,bid,2,2024.13,0.142,1780212719326,1780212719358,95,32,False,False,10670015615659
6,ETHUSDT,ask,1,2024.15,291.381,1780212719326,1780212719358,95,32,False,False,10670015615659
7,ETHUSDT,ask,2,2024.16,0.061,1780212719326,1780212719358,95,32,False,False,10670015615659


## 5. Inspect Full Top-N Snapshots

In [5]:
def snapshot_to_frame(snapshot):
    rows = []
    for side_name, levels in [("bid", snapshot.bids), ("ask", snapshot.asks)]:
        for level in levels:
            rows.append(
                {
                    "symbol": snapshot.symbol,
                    "side": side_name,
                    "level": level.level,
                    "price": level.price,
                    "qty": level.qty,
                    "spread": snapshot.spread,
                    "spread_bps": snapshot.spread_bps,
                    "mid_price": snapshot.mid_price,
                    "final_update_id": snapshot.final_update_id,
                }
            )
    return pd.DataFrame(rows)


for symbol in config.symbols:
    snapshot = service.get_latest(symbol)
    if snapshot is None:
        print(symbol, "has no snapshot yet")
        continue
    print(symbol, "stale=", snapshot.is_stale, "sequence_gap=", snapshot.sequence_gap)
    display(snapshot_to_frame(snapshot))

BTCUSDT stale= False sequence_gap= False


,symbol,side,level,price,qty,spread,spread_bps,mid_price,final_update_id
0,BTCUSDT,bid,1,73865.60,0.587,0.10,0.01353809246923299260210937019,73865.65,10670015725559
1,BTCUSDT,bid,2,73865.50,0.004,0.10,0.01353809246923299260210937019,73865.65,10670015725559
2,BTCUSDT,bid,3,73865.40,0.001,0.10,0.01353809246923299260210937019,73865.65,10670015725559
3,BTCUSDT,bid,4,73865.30,0.106,0.10,0.01353809246923299260210937019,73865.65,10670015725559
4,BTCUSDT,bid,5,73865.10,0.008,0.10,0.01353809246923299260210937019,73865.65,10670015725559
5,BTCUSDT,ask,1,73865.70,10.872,0.10,0.01353809246923299260210937019,73865.65,10670015725559
6,BTCUSDT,ask,2,73865.80,0.010,0.10,0.01353809246923299260210937019,73865.65,10670015725559
7,BTCUSDT,ask,3,73865.90,0.003,0.10,0.01353809246923299260210937019,73865.65,10670015725559
8,BTCUSDT,ask,4,73866.00,0.007,0.10,0.01353809246923299260210937019,73865.65,10670015725559
9,BTCUSDT,ask,5,73866.10,0.056,0.10,0.01353809246923299260210937019,73865.65,10670015725559


ETHUSDT stale= False sequence_gap= False


,symbol,side,level,price,qty,spread,spread_bps,mid_price,final_update_id
0,ETHUSDT,bid,1,2024.14,166.423,0.01,0.04940357533674712038910255935,2024.145,10670015734997
1,ETHUSDT,bid,2,2024.13,0.142,0.01,0.04940357533674712038910255935,2024.145,10670015734997
2,ETHUSDT,bid,3,2024.12,25.023,0.01,0.04940357533674712038910255935,2024.145,10670015734997
3,ETHUSDT,bid,4,2024.11,0.011,0.01,0.04940357533674712038910255935,2024.145,10670015734997
4,ETHUSDT,bid,5,2024.10,0.056,0.01,0.04940357533674712038910255935,2024.145,10670015734997
5,ETHUSDT,ask,1,2024.15,263.443,0.01,0.04940357533674712038910255935,2024.145,10670015734997
6,ETHUSDT,ask,2,2024.16,0.061,0.01,0.04940357533674712038910255935,2024.145,10670015734997
7,ETHUSDT,ask,3,2024.18,0.013,0.01,0.04940357533674712038910255935,2024.145,10670015734997
8,ETHUSDT,ask,4,2024.19,0.001,0.01,0.04940357533674712038910255935,2024.145,10670015734997
9,ETHUSDT,ask,5,2024.20,0.031,0.01,0.04940357533674712038910255935,2024.145,10670015734997


## 6. Watch Updates Briefly

This cell samples the in-memory latest table every second for 10 seconds. It does not write CSV, Parquet, or database records.

In [6]:
for _ in range(10):
    sample_time_ms = time.time_ns() // 1_000_000
    rows = []
    for symbol in config.symbols:
        snapshot = service.get_latest(symbol)
        if snapshot is None:
            rows.append({"symbol": symbol, "state": "missing"})
            continue
        rows.append(
            {
                "symbol": symbol,
                "state": "stale" if snapshot.is_stale else "live",
                "sequence_gap": snapshot.sequence_gap,
                "bid1": snapshot.bids[0].price if snapshot.bids else None,
                "ask1": snapshot.asks[0].price if snapshot.asks else None,
                "spread_bps": snapshot.spread_bps,
                "quote_age_ms": sample_time_ms - snapshot.event_time_ms,
                "final_update_id": snapshot.final_update_id,
            }
        )
    display(pd.DataFrame(rows))
    time.sleep(1.0)

,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73860.50,73860.60,0.01353902726150834241012285990,57,10670016122266
1,ETHUSDT,live,False,2024.14,2024.15,0.04940357533674712038910255935,43,10670016124676


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73860.50,73860.60,0.01353902726150834241012285990,185,10670016208504
1,ETHUSDT,live,False,2024.14,2024.15,0.04940357533674712038910255935,225,10670016205611


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73860.50,73860.60,0.01353902726150834241012285990,133,10670016289487
1,ETHUSDT,live,False,2024.14,2024.15,0.04940357533674712038910255935,203,10670016284717


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73860.50,73860.60,0.01353902726150834241012285990,120,10670016349111
1,ETHUSDT,live,False,2024.14,2024.15,0.04940357533674712038910255935,40,10670016356221


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73860.50,73860.60,0.01353902726150834241012285990,130,10670016416393
1,ETHUSDT,live,False,2024.14,2024.15,0.04940357533674712038910255935,86,10670016419238


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73860.50,73860.60,0.01353902726150834241012285990,104,10670016481359
1,ETHUSDT,live,False,2024.14,2024.15,0.04940357533674712038910255935,120,10670016480459


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73860.50,73860.60,0.01353902726150834241012285990,78,10670016574359
1,ETHUSDT,live,False,2024.14,2024.15,0.04940357533674712038910255935,88,10670016573399


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73860.50,73860.60,0.01353902726150834241012285990,123,10670016654815
1,ETHUSDT,live,False,2024.12,2024.13,0.04940406348422157722472673377,81,10670016658227


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73860.50,73860.60,0.01353902726150834241012285990,133,10670016743278
1,ETHUSDT,live,False,2024.12,2024.13,0.04940406348422157722472673377,97,10670016745345


,symbol,state,sequence_gap,bid1,ask1,spread_bps,quote_age_ms,final_update_id
0,BTCUSDT,live,False,73860.50,73860.60,0.01353902726150834241012285990,90,10670016811889
1,ETHUSDT,live,False,2024.12,2024.13,0.04940406348422157722472673377,90,10670016811837


## 7. Stop The Background Thread

Always stop the service when you are done with the notebook kernel or before re-running the start cell.

In [7]:
service.stop(timeout=10.0)
service.status()

DepthServiceStatus(symbols=('BTCUSDT', 'ETHUSDT'), running=False, ready=False, connected=False, url='wss://fstream.binance.com/public/stream?streams=btcusdt@depth5@100ms/ethusdt@depth5@100ms', last_connect_at=datetime.datetime(2026, 5, 31, 7, 31, 57, 719637, tzinfo=datetime.timezone.utc), last_message_at=datetime.datetime(2026, 5, 31, 7, 32, 13, 476642, tzinfo=datetime.timezone.utc), reconnect_attempts=1, last_error=None)